<a href="https://colab.research.google.com/github/alanapooler827/554Homework7/blob/main/ST554_HW6_Pooler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Alana Pooler
<br>
Purpose: Complete Homework 7

The purpose of this homework is to practice fitting MLR and logistic regression models (including penalized or regularized models). We will use the wine quality data set from the UCI Machine Learning Repository.

Input variables (based on physicochemical tests)

* fixed acidity
* volatile acidity
* citric acid
* residual sugar
* chlorides
* free sulfur dioxide
* total sulfur dioxide
* density
* pH
* sulphates
* alcohol


Output variable (based on sensory data): quality (score between 0 and 10)

Rather than try to predict quality, let's make our target variable for fitting multiple linear regression type models alcohol.

For fitting logistic regression type models we'll use the type of wine as the response variable.

First import some packages we will need throughout the notebook.

In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso

### Read in wine data sets

Read in the red wine data set and view the first few rows

In [30]:
red = pd.read_csv('winequality-red.csv', sep=';')
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Read in the white wine data set and view the first few rows

In [31]:
white = pd.read_csv('winequality-white.csv', sep=';')
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


Before combining the two data sets together, we will add a 'type' column to each to label the wines as 'red' and 'white

In [32]:
red['type'] = 'red'
white['type'] = 'white'

Now we will combine the two data sets together and view the first few rows of the new data set.

In [33]:
wine = pd.concat([red, white])
wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


### Split the Data

Now we need to split up the data set into a training and test set. We will use stratified sampling to make sure that there is a similar proportion of white and red wines in the training and test sets.

We will use 80% of the data for the training set and 20% of the data for the test set.

In [34]:
df_train, df_test = train_test_split(
    wine,
    test_size=0.2,
    stratify=wine['type'],
    random_state=42
)

# create X and Y train sets for regression
y_train_reg = df_train['alcohol']
X_train_reg = df_train.drop(columns=['alcohol', 'type'])

# create X and Y test sets for regression
y_test_reg = df_test['alcohol']
X_test_reg = df_test.drop(columns=['alcohol', 'type'])

Before we start building any models, let's look at correlations between variables so that we can get a better idea of the data and which fields will be useful predictors.

We will look at the correlation between our response variable, alcohol, and the other numeric variables.

The variables density and quality have a decently strong correlation with alcohol. The variables total sulfar dioxide, residual sugar, chlorides, and free sulfur dioxides have stronger correlations with alcohol than the rest of the variables, but the correlations are pretty weak.

In [101]:
wine.corr(numeric_only=True)["alcohol"]

,alcohol
fixed acidity,-0.095452
volatile acidity,-0.037640
citric acid,-0.010493
residual sugar,-0.359415
chlorides,-0.256916
free sulfur dioxide,-0.179838
total sulfur dioxide,-0.265740
density,-0.686745
pH,0.121248
sulphates,-0.003029


Let's also create a basic MLR model without using cross validation so that we can retrieve variable importance.

This shows that density is the most important predictor by far. Some of the variables that have higher correlations with alcohol have lower importance, like total sulfur dioxide. Based on these results, we will include


In [102]:
model = LinearRegression()
model.fit(X_train_reg, y_train_reg)

importance = pd.DataFrame({'col': X_train_reg.columns, 'importance': abs(model.coef_)})
importance.sort_values(by='importance', ascending=False)

,col,importance
7,density,573.569121
8,pH,2.760674
1,volatile acidity,1.547387
9,sulphates,1.331144
0,fixed acidity,0.524955
2,citric acid,0.458453
4,chlorides,0.245192
3,residual sugar,0.193377
10,quality,0.137141
6,total sulfur dioxide,0.004032


## Regression Task

We will train several regression models - multiple linear regression, LASSO, ridge regression, and elastic net. For every model, we will use 'alcohol' as the response variable.

First, we will fit four different multiple linear regression models and use CV to select the best one.
* At least one should include interaction terms
* At least one should include some polynomial terms (you may want to standardize your predictors
but that is up to you)
* Use CV to select your best MLR model

The first MLR model will be a full model using all variables in the data set (except for the response variable) as predictors.

In [186]:
# Full model
mlr_full_model = cross_validate(
    LinearRegression(),
    X_train_reg,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error",
    return_estimator=True)

The second will be a reduced model, using only a subset of the columns in the wine data set. I have chosen this subset columns using the correlations and variable importance we looked at earlier.

In [187]:
# create train and test sets containing selected subset of variables
subset = ['density', 'residual sugar', 'pH', 'volatile acidity', 'sulphates', 'fixed acidity', 'total sulfur dioxide']
X_train_subset = X_train_reg[subset]
X_test_subset = X_test_reg[subset]

mlr2 = cross_validate(
    LinearRegression(),
    X_train_subset,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error")

The third model will incorporate interaction terms. We will use the same subset of predictors we used in the second model.

Since sugar contributes to the density of the wine, these variables may be dependent on each other, so we will create an interaction term between residual sugar and density.

I also read on google that volatile acidity can have an impact on density, so I included an interaction term for those variables as well.

In [188]:
# create copies of training and test data
X_train_mlr3 = X_train_subset.copy()
X_test_mlr3 = X_test_subset.copy()

# add new interaction term to training and test data
X_train_mlr3["sugar_x_density"] = (
    X_train_mlr3["residual sugar"] * X_train_mlr3["density"]
)

X_test_mlr3["sugar_x_density"] = (
    X_test_mlr3["residual sugar"] * X_test_mlr3["density"]
)

X_train_mlr3["va_x_density"] = (
    X_train_mlr3["volatile acidity"] * X_train_mlr3["density"]
)

X_test_mlr3["va_x_density"] = (
    X_test_mlr3["volatile acidity"] * X_test_mlr3["density"]
)

# fit model 3
mlr3 = cross_validate(
    LinearRegression(),
    X_train_mlr3,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error")

The fourth and final MLR model will use polynomial terms. We will use the same subset of predictors we used in the second model. We will square the density and residual sugar variables to create polynomial terms, since these variables had the highest correlations with alcohol.

In [189]:
# create copy of training and test sets
X_train_mlr4 = X_train_subset.copy()
X_test_mlr4 = X_test_subset.copy()

# create squared density variable in the training and test sets
X_train_mlr4["density_sq"] = (
    X_train_mlr4["density"] ** 2
)

X_test_mlr4["density_sq"] = (
    X_test_mlr4["density"] ** 2
)

# create squared residual sugar variable in the training and test sets
X_train_mlr4["residual_sugar_sq"] = (
    X_train_mlr4["residual sugar"] ** 2
)

X_test_mlr4["residual_sugar_sq"] = (
    X_test_mlr4["residual sugar"] ** 2
)

# fit model 4
mlr4 = cross_validate(
    LinearRegression(),
    X_train_mlr4,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error")

Now we will look at the scores from each of the four MLR models to identify the best one.

The model with the polynomial terms has the lowest root mean squared error out of the four models.

In [194]:
print('Full model RMSE:', round(np.sqrt(-sum(mlr_full_model['test_score'])),4))
print('Reduced model RMSE:', round(np.sqrt(-sum(mlr2['test_score'])),4))
print('Interaction model RMSE:', round(np.sqrt(-sum(mlr3['test_score'])),4))
print('Polynomial model RMSE:', round(np.sqrt(-sum(mlr4['test_score'])),4))

Full model RMSE: 1.2249
Reduced model RMSE: 1.2485
Interaction model RMSE: 1.2823
Polynomial model RMSE: 1.1238


Fit a LASSO model with a set of predictors of your choosing
* Use at least five predictors
* Use CV to select the tuning parameter

Fit a Ridge Regression model with a set of predictors of your choosing
* Use at least five predictors
* Use CV to select the tuning parameter

Fit an Elastic Net model with a set of predictors of your choosing
* Use at least five predictors
* Use CV to select the tuning parameters

## Classification Task

In [ ]:
# create X and Y train sets for classification
y_train_clf = (df_train['wine'] == 'white').astype(int)
X_train_clf = df_train.drop(columns=['wine'])

# create X and Y test sets for classification
y_test_clf = (df_test['wine'] == 'white').astype(int)
X_test_clf = df_test.drop(columns=['wine'])